# Full MEA Pipeline

Runs all six analysis stages end-to-end for every well in the dataset.

```
preprocessing → sorting → auto_merge → analyzer → auto_curation → burst_detection
```

| Stage | Task | Typical runtime |
|---|---|---|
| 1 | `preprocessing` | ~1–5 min / well |
| 2 | `sorting` | ~10–60 min / well (GPU-dependent) |
| 3 | `auto_merge` | seconds (disabled) / ~5 min (enabled) |
| 4 | `analyzer` | ~5–20 min / well |
| 5 | `auto_curation` | < 1 min / well |
| 6 | `burst_detection` | < 1 min / well |

**Configuration:** edit `pipeline_config.json` (generated on first run) to set paths,
sorting parameters, curation thresholds, and merge options.

**Re-runs are safe:** the pipeline cache skips completed wells automatically.
Changing any parameter in `pipeline_config.json` invalidates the affected task
and all downstream tasks for wells that have that config cached.

In [1]:
import sys
import time
import traceback
import logging
from pathlib import Path

import pandas as pd

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

from dataset_manager import DatasetManager
from pipeline_manager import PipelineManager, WorkItem
from pipeline_manager.task_record import TaskStatus
from pipeline_tasks import (
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
)
from config_manager import ConfigManager

logging.basicConfig(level=logging.WARNING)
print("Imports OK")

Imports OK


## Task chain

Inspect the registered task order and dependency graph before doing anything else.

In [2]:
ALL_TASK_CLASSES = [
    PreprocessingTask,
    SortingTask,
    AutoMergeTask,
    AnalyzerTask,
    AutoCurationTask,
    BurstDetectionTask,
]

TASK_ORDER = [cls.task_name for cls in ALL_TASK_CLASSES]

print("Pipeline execution order:")
for i, cls in enumerate(ALL_TASK_CLASSES, 1):
    deps = " → ".join(cls.dependencies) if cls.dependencies else "(none)"
    print(f"  {i}. {cls.task_name:<20s}  depends on: {deps}")

Pipeline execution order:
  1. preprocessing         depends on: (none)
  2. sorting               depends on: preprocessing
  3. auto_merge            depends on: sorting
  4. analyzer              depends on: auto_merge
  5. auto_curation         depends on: analyzer
  6. burst_detection       depends on: auto_curation


## Config

On the **first run** this cell writes `pipeline_config.json` with all task defaults and
stops — edit it to set `data_root`, `analysis_root`, and any parameters you want to
override, then re-run.

Key parameters to review before the first full run:

| Section | Key | What to set |
|---|---|---|
| `global` | `data_root` | Path to raw `.h5` recordings |
| `global` | `analysis_root` | Where all outputs are written |
| `sorting` | `sorter` | Sorter name (`kilosort4`) |
| `auto_merge` | `enabled` | `false` to skip merging |
| `auto_curation` | `enabled` | `false` to keep all units |
| `auto_curation` | `*_min` / `*_max` | Curation thresholds |

In [3]:
CONFIG_FILE = Path("../config/default_tasks_params.json")

cm = ConfigManager()
for cls in ALL_TASK_CLASSES:
    cm.register_task(cls)

if not CONFIG_FILE.exists():
    cm.generate_template(CONFIG_FILE)
    raise RuntimeError(
        f"Template written to {CONFIG_FILE}.\n"
        "Edit it — set data_root, analysis_root, and any parameters — "
        "then re-run this cell."
    )

cm.load(CONFIG_FILE)

DATA_ROOT     = Path(cm.get_global("data_root"))
ANALYSIS_ROOT = Path(cm.get_global("analysis_root"))

print(f"Config loaded: {CONFIG_FILE}")
print(f"  data_root:     {DATA_ROOT}")
print(f"  analysis_root: {ANALYSIS_ROOT}")
print()
for cls in ALL_TASK_CLASSES:
    params = cm.get_task_params(cls.task_name)
    print(f"  [{cls.task_name}]  {params}")

Config loaded: ../config/default_tasks_params.json
  data_root:     /mnt/benshalom-nas/raw_data/rbs_maxtwo_desktop/harddisk20tbV2/Sadegh_Lab/CX138
  analysis_root: ../data/analysis

  [preprocessing]  {'output_root': './preprocessed_data', 'bandpass_freq_min': 300, 'bandpass_freq_max': 3000, 'reference': 'local', 'operator': 'median', 'local_radius': [0, 250], 'dtype': 'float32', 'n_jobs': 30, 'chunk_duration': '1s', 'progress_bar': True, 'overwrite': True}
  [sorting]  {'preprocessing_output_root': './preprocessed_data', 'output_root': './spikesorted_data', 'sorter': 'kilosort4', 'docker_image': None, 'verbose': True, 'remove_existing_folder': True, 'delete_output_folder': False, 'overwrite': True, 'clean_excess_spikes': True, 'remove_empty_units': True, 'min_high_vram_gb': 14, 'high_vram_sorter_kwargs': {'batch_size_seconds': 2.0, 'clear_cache': True, 'invert_sign': True, 'cluster_downsampling': 20, 'max_cluster_subset': None, 'nblocks': 0, 'dmin': 17, 'do_correction': False}, 'low_v

## 1. Scan recordings

In [4]:
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

dataset_mgr = DatasetManager(DATA_ROOT, ANALYSIS_ROOT)
recordings  = dataset_mgr.get_recording_by([("scan_type", "==", "Network"),
                                            ("run_id", "==", "000029")])

print(f"Found {len(recordings)} Network recording(s)")

pd.DataFrame([
    {
        "recording_key": r.cache_key,
        "date":          r.date,
        "sample_id":     r.sample_id,
        "plate_id":      r.plate_id,
        "scan_type":     r.scan_type,
        "run_id":        r.run_id,
        "n_wells":       len(r.wells),
        "size_GB":       round(r.file_size / 1e9, 2),
    }
    for r in recordings
])

Found 1 Network recording(s)


,recording_key,date,sample_id,plate_id,scan_type,run_id,n_wells,size_GB
0,CX138/260329/T003346/Network/000029,260329,CX138,T003346,Network,000029,24,18.93


## 2. Register all tasks and add wells

In [5]:
pipeline_mgr = PipelineManager(ANALYSIS_ROOT, config_provider=cm)

for cls in ALL_TASK_CLASSES:
    try:
        pipeline_mgr.register_task(cls)
        print(f"  registered: {cls.task_name!r}")
    except ValueError as e:
        print(f"  already registered: {cls.task_name!r} ({e})")

n_added   = 0
n_skipped = 0
for rec in recordings:
    if not rec.h5_recordings:
        print(f"WARNING: no h5 structure for {rec.cache_key!r} — skipping")
        n_skipped += 1
        continue
    for rec_name, well_ids in rec.h5_recordings.items():
        for well_id in well_ids:
            pipeline_mgr.add_well(rec.cache_key, f"{rec_name}/{well_id}")
            n_added += 1

print(f"\nWork units registered: {n_added}  |  Recordings skipped: {n_skipped}")

  registered: 'preprocessing'
  registered: 'sorting'
  registered: 'auto_merge'
  registered: 'analyzer'
  registered: 'auto_curation'
  registered: 'burst_detection'

Work units registered: 24  |  Recordings skipped: 0


## 3. Pipeline status overview

The matrix below shows the current status of every (well × task) combination.
Run this cell at any time to check progress.

In [6]:
_STATUS_SYMBOL = {
    str(TaskStatus.NOT_RUN):  "\u2014",       # —
    str(TaskStatus.RUNNING):  "\u23f3",       # ⏳
    str(TaskStatus.COMPLETE): "\u2713",       # ✓
    str(TaskStatus.FAILED):   "\u2717",       # ✗
}


def _pipeline_matrix(mgr: PipelineManager) -> pd.DataFrame:
    rows = []
    for entry in mgr.entries:
        rec_name, well_id = entry.well_id.split("/", 1)
        row: dict = {
            "recording_key": entry.recording_key,
            "rec_name":      rec_name,
            "well_id":       well_id,
        }
        for task_name in TASK_ORDER:
            t = entry.tasks.get(task_name)
            status_str = str(t.status) if t else str(TaskStatus.NOT_RUN)
            row[task_name] = _STATUS_SYMBOL.get(status_str, status_str)
        rows.append(row)
    return pd.DataFrame(rows)


df_matrix = _pipeline_matrix(pipeline_mgr)
print("Legend:  \u2014 not run   \u23f3 running   \u2713 complete   \u2717 failed")
df_matrix

Legend:  — not run   ⏳ running   ✓ complete   ✗ failed


,recording_key,rec_name,well_id,preprocessing,sorting,auto_merge,analyzer,auto_curation,burst_detection
0,CX138/260327/T003346/Network/000019,rec0000,well000,✓,—,—,—,—,—
1,CX138/260327/T003346/Network/000019,rec0000,well001,✓,—,—,—,—,—
2,CX138/260327/T003346/Network/000019,rec0000,well002,✓,—,—,—,—,—
3,CX138/260327/T003346/Network/000019,rec0000,well003,✓,—,—,—,—,—
4,CX138/260327/T003346/Network/000019,rec0000,well004,✓,—,—,—,—,—
...,...,...,...,...,...,...,...,...,...
523,CX138/260404/T003346/Network/000044,rec0003,well019,—,—,—,—,—,—
524,CX138/260404/T003346/Network/000044,rec0003,well020,—,—,—,—,—,—
525,CX138/260404/T003346/Network/000044,rec0003,well021,—,—,—,—,—,—
526,CX138/260404/T003346/Network/000044,rec0003,well022,—,—,—,—,—,—


## 4. Run full pipeline

`PipelineManager.get_next_task()` respects dependency order — it only returns a
task once all its upstream tasks are complete for that well.  The loop below runs
whatever is ready next until nothing remains.

**To selectively retry failed tasks** add their names to `RETRY_FAILED_TASKS`:
```python
RETRY_FAILED_TASKS = {"analyzer", "auto_curation"}
```

**To stop after a specific stage** set `STOP_AFTER_TASK`:
```python
STOP_AFTER_TASK = "sorting"   # pause before auto_merge
```

**To reset a specific well:** `pipeline_mgr.refresh(task_name, recording_key=..., well_id="rec0000/well000")`  
**To reset a full task across all wells:** `pipeline_mgr.refresh(task_name)`

In [ ]:
# ── Control knobs ────────────────────────────────────────────────────────────
RETRY_FAILED_TASKS: set[str] = set()   # e.g. {\"analyzer\", \"auto_curation\"}
STOP_AFTER_TASK:    str | None = None   # e.g. \"sorting\" to pause after sorting
# ─────────────────────────────────────────────────────────────────────────────

_task_instances: dict[str, object] = {cls.task_name: cls() for cls in ALL_TASK_CLASSES}
_rec_lookup:     dict[str, object] = {r.cache_key: r for r in recordings}
_completed_stages: list[str]       = []

# Scope get_next_task() to only the recordings loaded in this session.
# The pipeline cache may contain entries from previous runs with different
# recording keys; without this filter, those stale entries would be returned
# and fail the _rec_lookup lookup below.
_current_recording_keys = set(_rec_lookup.keys())

while True:
    work_items = pipeline_mgr.get_next_task(
        n=1, retry_failed=False, recording_keys=_current_recording_keys
    )

    # Also poll failed tasks that are explicitly marked for retry
    if not work_items and RETRY_FAILED_TASKS:
        work_items = [
            item
            for item in pipeline_mgr.get_next_task(
                n=1, retry_failed=True, recording_keys=_current_recording_keys
            )
            if item.task_name in RETRY_FAILED_TASKS
        ]

    if not work_items:
        break

    item:      WorkItem = work_items[0]
    task                = _task_instances[item.task_name]
    rec_entry           = _rec_lookup[item.recording_key]
    rec_name, well_id   = item.well_id.split("/", 1)
    params              = cm.get_task_params(item.task_name)

    print(f"\n[{item.task_name}]  {item.recording_key} / {rec_name} / {well_id}")
    pipeline_mgr.update_status(item, TaskStatus.RUNNING)

    t0 = time.perf_counter()
    try:
        output_path = task.run(
            item.recording_key,
            item.well_id,
            dataset_mgr.get_path(rec_entry),
            params,
        )
        elapsed = time.perf_counter() - t0
        pipeline_mgr.update_status(item, TaskStatus.COMPLETE, output_path=output_path)
        print(f"  ✓  {elapsed:.1f}s  → {output_path}")
        _completed_stages.append(item.task_name)
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        pipeline_mgr.update_status(item, TaskStatus.FAILED, error=str(exc))
        traceback.print_exc()
        print(f"  ✗  {elapsed:.1f}s  failed: {exc}")

    if STOP_AFTER_TASK and item.task_name == STOP_AFTER_TASK:
        print(f"\nPaused after {STOP_AFTER_TASK!r} as requested.")
        break

print("\n— No more pending tasks. —")


[auto_curation]  CX138/260329/T003346/Network/000029 / rec0000 / well000
  ✓  0.3s  → curation_data/CX138/260329/T003346/Network/000029/rec0000/well000/auto_curation

[burst_detection]  CX138/260329/T003346/Network/000029 / rec0000 / well000
  ✓  0.3s  → burst_detection_data/CX138/260329/T003346/Network/000029/rec0000/well000/burst_detection

[auto_curation]  CX138/260329/T003346/Network/000029 / rec0000 / well001
  ✓  0.1s  → curation_data/CX138/260329/T003346/Network/000029/rec0000/well001/auto_curation

[burst_detection]  CX138/260329/T003346/Network/000029 / rec0000 / well001
  ✓  0.0s  → burst_detection_data/CX138/260329/T003346/Network/000029/rec0000/well001/burst_detection

[auto_curation]  CX138/260329/T003346/Network/000029 / rec0000 / well002
  ✓  0.1s  → curation_data/CX138/260329/T003346/Network/000029/rec0000/well002/auto_curation

[burst_detection]  CX138/260329/T003346/Network/000029 / rec0000 / well002
  ✓  0.0s  → burst_detection_data/CX138/260329/T003346/Network/0000

estimate_sparsity (no parallelization):   0%|          | 0/301 [00:00<?, ?it/s]

compute_waveforms (no parallelization):   0%|          | 0/301 [00:00<?, ?it/s]

noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/template_metrics.py:301: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/templat

...

/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/template_metrics.py:301: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/template/metrics.py:936: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/metrics/templat

Compute : spike_amplitudes (no parallelization):   0%|          | 0/301 [00:00<?, ?it/s]

/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/spikeinterface/core/analyzer_extension_core.py:1165: UserWarning: The following metrics will not be computed due to missing dependencies: ['silhouette', 'mahalanobis', 'nearest_neighbor', 'drift', 'd_prime']
  warnings.warn(
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/mnt/Vol20tb1/yuxin/app/miniconda/envs/yuxin_mea/lib/python3.11/site-pa

  ✓  54.0s  → analyzer_data/CX138/260329/T003346/Network/000029/rec0001/well008/analyzer

[auto_curation]  CX138/260329/T003346/Network/000029 / rec0001 / well008
  ✓  0.1s  → curation_data/CX138/260329/T003346/Network/000029/rec0001/well008/auto_curation

[burst_detection]  CX138/260329/T003346/Network/000029 / rec0001 / well008
  ✓  0.0s  → burst_detection_data/CX138/260329/T003346/Network/000029/rec0001/well008/burst_detection

[preprocessing]  CX138/260329/T003346/Network/000029 / rec0001 / well009
The h5 compression library for Maxwell is already located in /home/yuxin/hdf5_plugin_path_maxwell/libcompression.so!
write_zarr_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=25.18 MiB - total_memory=755.31 MiB - chunk_duration=1.00s


write_zarr_recording (workers: 30 processes fork):   0%|          | 0/301 [00:00<?, ?it/s]

## 5. Final status

In [ ]:
df_final = _pipeline_matrix(pipeline_mgr)

print("=== Per-task summary ===")
for task_name in TASK_ORDER:
    col = df_final[task_name]
    counts = col.value_counts().to_dict()
    print(f"  {task_name:<20s}  {counts}")

failed_mask = (df_final[TASK_ORDER] == "\u2717").any(axis=1)
if failed_mask.any():
    print("\n=== Failed wells ===")
    for _, row in df_final[failed_mask].iterrows():
        failed_tasks = [t for t in TASK_ORDER if row[t] == "\u2717"]
        print(f"  {row['recording_key']} / {row['rec_name']} / {row['well_id']}")
        print(f"    failed stages: {failed_tasks}")

print("\nLegend:  \u2014 not run   \u23f3 running   \u2713 complete   \u2717 failed")
df_final

=== Per-task summary ===
  preprocessing         {'—': 515, '✓': 13}
  sorting               {'—': 520, '✓': 8}
  auto_merge            {'—': 520, '✓': 8}
  analyzer              {'—': 520, '✓': 7, '⏳': 1}
  auto_curation         {'—': 521, '✗': 7}
  burst_detection       {'—': 528}

=== Failed wells ===
  CX138/260329/T003346/Network/000029 / rec0000 / well000
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0000 / well001
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0000 / well002
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0000 / well003
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0000 / well004
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0000 / well005
    failed stages: ['auto_curation']
  CX138/260329/T003346/Network/000029 / rec0001 / well006
    failed stages: ['auto_curation']

Legend:  — not run   ⏳ runni

,recording_key,rec_name,well_id,preprocessing,sorting,auto_merge,analyzer,auto_curation,burst_detection
0,CX138/260327/T003346/Network/000019,rec0000,well000,✓,—,—,—,—,—
1,CX138/260327/T003346/Network/000019,rec0000,well001,✓,—,—,—,—,—
2,CX138/260327/T003346/Network/000019,rec0000,well002,✓,—,—,—,—,—
3,CX138/260327/T003346/Network/000019,rec0000,well003,✓,—,—,—,—,—
4,CX138/260327/T003346/Network/000019,rec0000,well004,✓,—,—,—,—,—
...,...,...,...,...,...,...,...,...,...
523,CX138/260404/T003346/Network/000044,rec0003,well019,—,—,—,—,—,—
524,CX138/260404/T003346/Network/000044,rec0003,well020,—,—,—,—,—,—
525,CX138/260404/T003346/Network/000044,rec0003,well021,—,—,—,—,—,—
526,CX138/260404/T003346/Network/000044,rec0003,well022,—,—,—,—,—,—


## 6. Curation summary

Reads `quality_metrics.parquet` from every completed `auto_curation` well
and produces an across-wells summary table.

In [1]:
summary_rows = []

for entry in pipeline_mgr.entries:
    curation_task = entry.tasks.get(AutoCurationTask.task_name)
    if not curation_task or curation_task.status != TaskStatus.COMPLETE:
        continue

    qm_path = Path(curation_task.output_path) / "quality_metrics.pkl"
    if not qm_path.exists():
        continue

    rec_name, well_id = entry.well_id.split("/", 1)
    qm = pd.read_pickle(qm_path)

    curated = qm[qm["curated"]]
    summary_rows.append({
        "recording_key": entry.recording_key,
        "rec_name":      rec_name,
        "well_id":       well_id,
        "n_total":        len(qm),
        "n_curated":      len(curated),
        "pct_kept":       round(100 * len(curated) / len(qm), 1) if len(qm) else 0,
        "median_fr_hz":   round(curated["firing_rate"].median(), 3)
                          if "firing_rate" in curated.columns and len(curated) else None,
        "median_amp_uv":  round(curated["amplitude_median"].median(), 1)
                          if "amplitude_median" in curated.columns and len(curated) else None,
    })

if summary_rows:
    df_curation = pd.DataFrame(summary_rows)
    print(f"Wells with curation results: {len(df_curation)}")
    print(f"Total curated units: {df_curation['n_curated'].sum()}  /  {df_curation['n_total'].sum()}")
    df_curation
else:
    print("No curation results yet — run the pipeline first.")

NameError: name 'pipeline_mgr' is not defined